# Component 2: Phase 7 baseline comparison (Colab)

Compares 5 models (threshold rules, logistic regression, decision tree, random forest, 8-16-5 tanh MLP) on 3 feature sets (physics8, physics7 without `v_deficit`, raw8), using:
- the Phase 4 run-wise CV folds (TRAINING runs only);
- leave-one-site-out within the TRAINING runs.

**The OPERATION window is not used.** That window is the final test and is run once, after the model is chosen.

All code comes from `component2-tinyml/src/tinyml/` (`experiment.py`, `models.py`, `evaluate.py`). This notebook only calls it.

Run the cells in order with **Shift + Enter**. Step 3 asks you to upload `pv_fault_dataset.csv`. Step 5 takes about 1–3 minutes on a CPU runtime.

## Step 1: Get the latest repository code and install dependencies

In [ ]:
# STEP 1: Get the latest component2-tinyml code and install dependencies
import os, sys, subprocess

REPO_URL = "https://github.com/Lisura123/J26-IT-348"
BRANCH = "component2-tinyml"
REPO_DIR = "/content/J26-IT-348"
COMPONENT_DIR = f"{REPO_DIR}/component2-tinyml"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("Commit:", commit)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{COMPONENT_DIR}/requirements.txt"], check=True)
import numpy as np, pandas as pd, sklearn
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__)

## Step 2: Import the repository modules

In [ ]:
# STEP 2: Import the repository modules
SRC_DIR = f"{COMPONENT_DIR}/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from tinyml.validate import file_sha256
from tinyml.splits import load_assignments
from tinyml.experiment import run_comparison, summary_table, ablation_table, PROTOCOLS, FEATURE_SETS
from tinyml.models import MODEL_NAMES
print("Protocols   :", PROTOCOLS)
print("Feature sets:", FEATURE_SETS)
print("Models      :", MODEL_NAMES)

## Step 3: Upload the dataset
Click **Choose Files** and select `pv_fault_dataset.csv`. The file stays only in this Colab session.

In [ ]:
# STEP 3: Upload pv_fault_dataset.csv
from google.colab import files
uploaded = files.upload()
csv_files = [n for n in uploaded if n.endswith(".csv")]
if not csv_files:
    raise SystemExit("No CSV uploaded. Run this cell again.")
DATA_PATH = os.path.abspath(csv_files[0])
print("Dataset file:", DATA_PATH, "|", os.path.getsize(DATA_PATH), "bytes")

## Step 4: Check the dataset fingerprint and load the Phase 4 splits

In [ ]:
# STEP 4: SHA-256 check and saved run-wise splits
SPLITS_PATH = f"{COMPONENT_DIR}/results/splits_runwise_v1.json"
df = pd.read_csv(DATA_PATH)
sha = file_sha256(DATA_PATH)
assignments, meta = load_assignments(SPLITS_PATH, df, dataset_sha256=sha)   # stops if the file differs
print("SHA-256 OK:", sha)
print("Splits:", meta["split_version"], "| seed", meta["seed"], "| runs", meta["runs"])

## Step 5: Run the baseline comparison (about 1–3 minutes)
Trains every model on each training fold and evaluates on the matching validation fold. Stochastic models (forest, MLP) use seeds 0, 1 and 2.

In [ ]:
# STEP 5: Run the comparison (TRAINING window only; no OPERATION data)
import warnings, time
t0 = time.time()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result = run_comparison(df, assignments)
result["dataset_sha256"] = sha
result["split_version"] = meta["split_version"]
result["commit"] = commit
result["warnings"] = sorted({f"{w.category.__name__}: {str(w.message)[:160]}" for w in caught})
print(f"{len(result['records'])} fits in {time.time() - t0:.0f} s")
print("Warnings:", result["warnings"] or "none")

## Step 6: Results tables
Reading-level metrics (pooled over the combined held-out predictions, plus fold mean/min), run-level metrics (majority vote per run), the `v_deficit` ablation (Finding B), and the classes present in each held-out partition. Recall is not reported for classes absent from a partition.

In [ ]:
# STEP 6: Reading-level, run-level, ablation and class-presence tables
pd.set_option("display.width", 220)
table = summary_table(result)
read_cols = ["feature_set", "model", "pooled_macro_f1", "macro_f1_mean", "macro_f1_std", "macro_f1_min",
             "pooled_open_recall", "pooled_short_recall", "size", "fits"]
run_cols = ["feature_set", "model", "runs", "run_vote_acc", "worst_run_acc"]
for protocol in result["protocols"]:
    t = table[table.protocol == protocol].sort_values(["feature_set", "pooled_macro_f1"], ascending=[True, False])
    print(f"=== {protocol}: READING-LEVEL ===")
    print(t[read_cols].round(3).to_string(index=False)); print()
    print(f"=== {protocol}: RUN-LEVEL (majority vote per run) ===")
    print(t[run_cols].round(3).to_string(index=False)); print()
    print(f"=== {protocol}: v_deficit ablation (pooled macro-F1) ===")
    print(ablation_table(table, protocol).round(4).to_string()); print()
    print(f"=== {protocol}: classes present in each held-out partition ===")
    for tc in result["test_classes"]:
        if tc["protocol"] == protocol:
            print(f"  {tc['fold']:12s} runs {tc['runs']:2d}  classes {tc['classes_in_partition']}")
    print()

## Step 7: Coverage (skipped readings are reported, not hidden)

In [ ]:
# STEP 7: Coverage per evaluated fold
seen = set()
for r in result["records"]:
    if (r["protocol"], r["fold"]) in seen:
        continue
    seen.add((r["protocol"], r["fold"]))
    c = r["coverage"]
    print(f"{r['protocol']:13s} {r['fold']:12s} total {c['total']:4d} classified {c['classified']:4d} "
          f"skipped {c['skipped']:3d} {c['skip_reasons'] or ''} {list(c['runs_not_fully_classified']) or ''}")

## Step 8: Save and download the results
Downloads `baselines_phase7.json` (every fit) and `baselines_phase7.md` (the tables). **Send both files back for review.**

In [ ]:
# STEP 8: Save and download results
import json, importlib.util
spec = importlib.util.spec_from_file_location("baselines", f"{COMPONENT_DIR}/scripts/04_compare_baselines.py")
baselines = importlib.util.module_from_spec(spec); spec.loader.exec_module(baselines)
with open("/content/baselines_phase7.json", "w") as f:
    json.dump(result, f, indent=1)
with open("/content/baselines_phase7.md", "w") as f:
    f.write(baselines.render(result, table, sha))
from google.colab import files
files.download("/content/baselines_phase7.json")
files.download("/content/baselines_phase7.md")
print("Saved and downloaded.")